# Limpeza do dataset Spotify

Notebook separado para remover registros considerados ruídos ou fora do escopo da análise.

Critérios aplicados:
- Faixas com duração inferior a 1 minuto
- Faixas sem BPM (0 BPM)
- Faixas com speechiness maior que 0,5
- Faixas com baixa popularidade (popularity <= 10)
- Faixas com baixa loudness (loudness <= -16)
- Faixas cujo nome contém termos como "rain sounds", "white noise" e outros ruídos sonoros semelhantes
- Remoção de faixas duplicadas, mantendo a versão com maior popularidade para reduzir a influência de compilações genéricas

In [4]:
# Instala pandas caso ainda não exista
try:
    import pandas as pd
except ModuleNotFoundError:
    !pip install pandas
    import pandas as pd

from pathlib import Path
from datetime import datetime

base_dir = Path.cwd()
csv_path = base_dir.parent / 'dataset' / 'dataset.csv'
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = base_dir.parent / 'dataset' / f'dataset_cleaned_{timestamp}.csv'

print(f'Arquivo de entrada: {csv_path}')
print(f'Arquivo de saída: {output_path}')


Arquivo de entrada: /home/gabe/re/r.ia-spotify_nano_challenge/dataset/dataset.csv
Arquivo de saída: /home/gabe/re/r.ia-spotify_nano_challenge/dataset/dataset_cleaned_20260902_083357.csv


In [5]:
df = pd.read_csv(csv_path)
print('Shape original:', df.shape)
print(df.head(3).to_string(index=False))
print('\nColunas:', list(df.columns))

Shape original: (114000, 21)
 Unnamed: 0               track_id                artists       album_name       track_name  popularity  duration_ms  explicit  danceability  energy  key  loudness  mode  speechiness  acousticness  instrumentalness  liveness  valence  tempo  time_signature track_genre
          0 5SuOikwiRyPMVoIQDJUgSV            Gen Hoshino           Comedy           Comedy          73       230666     False         0.676   0.461    1    -6.746     0       0.1430        0.0322          0.000001     0.358    0.715 87.917               4    acoustic
          1 4qPNDBW1i3p13qLCt0Ki3A           Ben Woodward Ghost (Acoustic) Ghost - Acoustic          55       149610     False         0.420   0.166    1   -17.235     1       0.0763        0.9240          0.000006     0.101    0.267 77.489               4    acoustic
          2 1iJBSr7s7jYXzM8EGcbK5b Ingrid Michaelson;ZAYN   To Begin Again   To Begin Again          57       210826     False         0.438   0.359    0    -9.734 

In [6]:
noise_keywords = [
    'rain sounds', 'white noise', 'nature sounds', 'sleep', 'meditation',
    'ambient', 'environmental sounds', 'soothing', 'soundscape'
]

# Normalização
# 1) remover colunas inúteis, converter textos e padronizar nomes

df = df.copy()
for column in ['track_name', 'artists', 'album_name', 'track_genre']:
    if column in df.columns:
        df[column] = df[column].fillna('').astype(str).str.strip()

df['track_genre'] = df['track_genre'].str.lower()
df['track_name'] = df['track_name'].str.lower()
df['artists'] = df['artists'].str.lower()

# 2) resolver faixas duplicadas mantendo a versão com maior popularidade
# Isso ajuda a excluir compilações genéricas e versões repetidas do mesmo tema
original_len = len(df)
df['track_key'] = df['track_name'] + ' | ' + df['artists']
df = (
    df.sort_values(['track_key', 'popularity'], ascending=[True, False], na_position='last')
      .drop_duplicates(subset='track_key', keep='first')
      .drop(columns=['track_key'])
)
print(f'Faixas duplicadas removidas: {original_len - len(df)}')

# Cria máscara de exclusão
mask = pd.Series(False, index=df.index)

mask |= df['duration_ms'] < 60000
mask |= df['tempo'] == 0
mask |= df['speechiness'] > 0.5
mask |= df['popularity'] <= 10
mask |= df['loudness'] <= -16

for keyword in noise_keywords:
    mask |= df['track_name'].str.contains(keyword, case=False, na=False)

# Totais por critério
criteria = {
    'duration_ms < 60000': df['duration_ms'] < 60000,
    'tempo == 0': df['tempo'] == 0,
    'speechiness > 0.5': df['speechiness'] > 0.5,
    'popularity <= 10': df['popularity'] <= 10,
    'loudness <= -16': df['loudness'] <= -16,
    'track_name contains noise keyword': False,
}

noise_mask = pd.Series(False, index=df.index)
for keyword in noise_keywords:
    noise_mask |= df['track_name'].str.contains(keyword, case=False, na=False)
criteria['track_name contains noise keyword'] = noise_mask

for name, cond in criteria.items():
    print(f'{name}: {int(cond.sum())} faixas removidas')

print(f'\nTotal de faixas removidas: {int(mask.sum())}')
print(f'Total restante: {int((~mask).sum())}')


Faixas duplicadas removidas: 32793
duration_ms < 60000: 742 faixas removidas
tempo == 0: 145 faixas removidas
speechiness > 0.5: 1117 faixas removidas
popularity <= 10: 8899 faixas removidas
loudness <= -16: 6762 faixas removidas
track_name contains noise keyword: 369 faixas removidas

Total de faixas removidas: 15725
Total restante: 65482


In [7]:
cleaned_df = df.loc[~mask].copy()
print('Shape final:', cleaned_df.shape)
print(cleaned_df.head(5).to_string(index=False))

Shape final: (65482, 21)
 Unnamed: 0               track_id           artists                          album_name                             track_name  popularity  duration_ms  explicit  danceability  energy  key  loudness  mode  speechiness  acousticness  instrumentalness  liveness  valence   tempo  time_signature track_genre
      36750 0fROT4kK5oTm8xO8PX6EJF             rilès                      !I'll Be Back!                         !i'll be back!          52       178533      True         0.823   0.612    1    -7.767     1       0.2480      0.168000           0.00000    0.1090    0.688 142.959               4      french
      92751 1hH0t381PIXmUVWyG1Vj3p      brian hyland                   The Bashful Blond                    "a" you're adorable          39       151680     False         0.615   0.375    0   -10.362     0       0.0319      0.482000           0.00000    0.1110    0.922 110.720               4  rockabilly
      66970 1B45DvGMoFWdbAEUH2qliG little apple band The 

In [8]:
if output_path.exists():
    output_path.unlink()

cleaned_df.to_csv(output_path, index=False)
print(f'Arquivo salvo em: {output_path}')
print(f'Novo arquivo criado: {output_path.exists()}')


Arquivo salvo em: /home/gabe/re/r.ia-spotify_nano_challenge/dataset/dataset_cleaned_20260902_083357.csv
Novo arquivo criado: True


## Resultado da limpeza

Este notebook remove as faixas que se enquadram nos critérios definidos e salva um arquivo limpo em `dataset/dataset_cleaned.csv`.

Você pode usar esse dataset limpo em consultas SQL, análise exploratória ou treino de modelos.